[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nasaharvest/crop-stage-detection/blob/main/examples/01_byod.ipynb)


In [ ]:
!git clone https://github.com/nasaharvest/crop-stage-detection.git /content/crop-stage-detection
%cd /content/crop-stage-detection/examples
!pip install -q -r ../requirements.txt


# Example 1 — Bring Your Own NDVI Data

No Google Earth Engine required. Load any NDVI time-series and estimate
the current crop stage.

### Input requirements

Your DataFrame should contain at least:

- a `date` column with datetime-like values
- an `NDVI` column with float values in the range $[-1, 1]$

Observations can be irregularly spaced (e.g., every 5–12 days from satellite).
The library resamples to daily resolution internally.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

from crop_stage import run_crop_stage_from_dataframe

## 1. Load sample data

In [ ]:
df_all = pd.read_csv("../sample_data/sample_ndvi.csv", parse_dates=["date"])
print(f"CSV contains {df_all['field_id'].nunique()} field(s): {sorted(df_all['field_id'].unique())}")

# Sections 1–3 walk through a single field end-to-end.
# Section 4 uses df_all to demonstrate batch processing.
df = df_all[df_all["field_id"] == "field_001"].reset_index(drop=True)
print(f"\nfield_001: {len(df)} observations, {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## 2. One-liner: smooth + estimate

In [ ]:
result = run_crop_stage_from_dataframe(df)   # id_col=None → single field → dict
for k, v in result.items():
    print(f"  {k:20s}: {v}")

## 3. Visualize the smoothed NDVI curve

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from crop_stage import smooth_daily_interpolate_ndvi

df_smooth = smooth_daily_interpolate_ndvi(df)

fig, ax = plt.subplots(figsize=(10, 4))

ax.scatter(df["date"], df["NDVI"], color="gray", alpha=0.6, s=40, label="Raw NDVI", zorder=3)
ax.plot(df_smooth["date"], df_smooth["NDVI_smooth"], color="steelblue", lw=2, label="Smoothed NDVI")

ax.axhline(result["Lower_threshold"], color="orange", ls="--", label=f"Lower threshold ({result['Lower_threshold']:.2f})")
ax.axhline(result["Upper_threshold"], color="green",  ls="--", label=f"Upper threshold ({result['Upper_threshold']:.2f})")

last_obs = df.sort_values("date").iloc[-1]
ax.scatter(last_obs["date"], last_obs["NDVI"], s=200, facecolors="none", edgecolors="red",
           linewidths=2, zorder=4, label="Current observation")

ax.set_ylim(0, 1)
ax.set_ylabel("NDVI")

ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(mdates.AutoDateLocator()))

ax.legend(loc="upper left")
ax.set_title(f"Stage: {result['Stage']} — {result['Stage_description']}")
plt.tight_layout()
plt.show()

## 4. Multiple fields at once

Pass `id_col` to process all fields in a single call. The result is a DataFrame
with one row per field.

The sample CSV contains two fields with deliberately different phenological stages:

- **field_001** — full season (March → November), currently post-harvest → **Stage E**
- **field_002** — truncated series (March → late July, NDVI still near peak) → **Stage C**

In [ ]:
results_df = run_crop_stage_from_dataframe(df_all, id_col="field_id")
cols = ["field_id", "crop_stage", "stage_description", "peak_date", "days_since_peak", "last_date"]
results_df[cols]

## 5. Tune the thresholds (optional)

You can adjust the threshold settings to better match your series:

In [ ]:
from crop_stage import estimate_stage_adaptive

df_smooth = smooth_daily_interpolate_ndvi(df)
result_tuned = estimate_stage_adaptive(
    df_smooth["NDVI_smooth"].to_numpy(),
    dates=df_smooth["date"],
    upper_percentile=75,   # less strict peak detection
    min_peak_ndvi=0.40,    # lower floor for short-stature crops
    lower_threshold=0.30,
)
print(result_tuned["Stage"], "—", result_tuned["Stage_description"])